In [1]:
## init mongo db and fiftyone connection
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [2]:
import fiftyone.brain as fob
from sklearn.preprocessing import normalize
import plotly.express as px
import skdim
import random
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import ot
from sklearn.manifold import TSNE
import cv2
from fiftyone import ViewField as F
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import random

# ## renders plotly properly in a html instance. 
# import plotly.io as pio
# pio.renderers.default = "notebook"


In [3]:
## Load dataset and views from mongodb 
dataset = fo.load_dataset("dugong")

## load the views
nc_view = dataset.load_saved_view("New_Caledonia")
wp_view = dataset.load_saved_view("West_Papua")

## create roi

In [22]:

def add_roi_grid(dataset, tile_size=640, overlap=100):
    for sample in dataset:
        w = sample.metadata.width
        h = sample.metadata.height

        rois = []

        # Calculate stride
        stride = tile_size - overlap

        # Create overlapping grid coordinates
        for y in range(0, h, stride):
            for x in range(0, w, stride):
                # Ensure we don't go out of bounds
                x_end = min(x + tile_size, w)
                y_end = min(y + tile_size, h)

                # If the tile would be smaller than tile_size, adjust the start
                # to keep the tile size consistent
                x_start = x_end - tile_size if x_end - tile_size >= 0 else 0
                y_start = y_end - tile_size if y_end - tile_size >= 0 else 0

                # relative coordinates [top_left_x, top_left_y, width, height]
                rois.append(
                    fo.Detection(
                        label=f"tile_{y}_{x}",
                        bounding_box=[
                            x_start / w,
                            y_start / h,
                            tile_size / w,
                            tile_size / h
                        ]
                    )
                )

        sample["roi_grid"] = fo.Detections(detections=rois)
        sample.save()

# Apply the grid to your dataset
add_roi_grid(dataset, tile_size=640, overlap=100)

In [23]:
next(iter(dataset))

<Sample: {
    'id': '69add898cfd942e1c6b5d3ad',
    'media_type': 'image',
    'filepath': '/share/home/e2406743/dataset/dataset/NC/Flight_226/images/GH034226-619fa2d56d4d3_92.jpeg',
    'tags': ['Flight_226', 'NC', 'NC', 'train'],
    'metadata': <ImageMetadata: {
        'size_bytes': 712381,
        'mime_type': 'image/jpeg',
        'width': 2704,
        'height': 1520,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 3, 8, 20, 14, 16, 713000),
    'last_modified_at': datetime.datetime(2026, 4, 7, 15, 23, 48, 43476),
    'region': 'NC',
    'subregion': 'NC',
    'mission_name': 'Flight_226',
    'sea_state': 0,
    'turbidity_global': 1,
    'turbidity_local': 'no',
    'sun_glitter': '0-0',
    'cloud_reflection': '0-0',
    'habitat_type': 'coral',
    'background_complexity': 'high',
    'coral': 'P',
    'sand': 'A',
    'dense_seagrass': 'A',
    'open_sea': 'P',
    'sparse_seagrass': 'A',
    'ground_truth': <Detections: {
        'detections':

In [17]:
# On a cluster: auto=False prevents it trying to open a browser
# Use port forwarding: ssh -L 5151:localhost:5151 user@cluster
session = fo.launch_app(dataset,
                        port=5151,
                        auto=False)
print(session.url)  

Session launched. Run `session.show()` to open the App in a cell output.
http://localhost:5151/


## add and translate bbox inside roi_grid

In [30]:
# dataset.delete_sample_field("roi_grid.detections.slicing_check")
# dataset.delete_sample_field("roi_grid.detections.slicing_check_id")
# dataset.delete_sample_field("roi_grid.detections.slicing_eval")

dataset.delete_sample_field("roi_grid.detections.bbox_translated")
#dataset.delete_sample_field("roi_grid.detections.contains_dugong")

In [21]:
dataset.delete_sample_field("roi_grid")

In [29]:
next(iter(dataset.skip(10).take(1)))

<SampleView: {
    'id': '69add89acfd942e1c6b5dce6',
    'media_type': 'image',
    'filepath': '/share/home/e2406743/dataset/dataset/WP/UM/UM_M5/images/MAN_P4_UM_M5_F2_GSP_DJI_0222-657c5a5b59a90_73.jpeg',
    'tags': ['UM_M5', 'UM', 'WP', 'fold_3', 'train'],
    'metadata': <ImageMetadata: {
        'size_bytes': 1186355,
        'mime_type': 'image/jpeg',
        'width': 4096,
        'height': 2160,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 3, 8, 20, 14, 18, 699000),
    'last_modified_at': datetime.datetime(2026, 4, 7, 15, 40, 0, 340000),
    'region': 'WP',
    'subregion': 'UM',
    'mission_name': 'UM_M5',
    'sea_state': 1,
    'turbidity_global': 1,
    'turbidity_local': 'Unknown',
    'sun_glitter': '25-50',
    'cloud_reflection': '0-0',
    'habitat_type': 'sand',
    'background_complexity': 'medium',
    'coral': 'P',
    'sand': 'P',
    'dense_seagrass': 'P',
    'open_sea': 'A',
    'sparse_seagrass': 'A',
    'ground_truth': <Dete

In [28]:

def clip_bbox(gt_bbox, tile_bbox):
    gx, gy, gw, gh = gt_bbox
    tx, ty, tw, th = tile_bbox

    # Calculate absolute coordinates
    gx1, gy1 = gx, gy
    gx2, gy2 = gx + gw, gy + gh
    tx1, ty1 = tx, ty
    tx2, ty2 = tx + tw, ty + th

    # Clip the bbox to the tile
    x1 = max(gx1, tx1)
    y1 = max(gy1, ty1)
    x2 = min(gx2, tx2)
    y2 = min(gy2, ty2)

    # If no overlap, return None
    if x1 >= x2 or y1 >= y2:
        return None

    # Return the clipped bbox in absolute coordinates
    return [x1, y1, x2 - x1, y2 - y1]

def normalize_bbox(clipped_bbox, tile_bbox):
    if clipped_bbox is None:
        return None
    x, y, w, h = clipped_bbox
    tx, ty, tw, th = tile_bbox

    # Normalize relative to the tile
    nx = (x - tx) / tw
    ny = (y - ty) / th
    nw = w / tw
    nh = h / th

    return [nx, ny, nw, nh]


def bbox_overlaps(gt_bbox, tile_bbox):
    gx, gy, gw, gh = gt_bbox
    tx, ty, tw, th = tile_bbox
    return not (
        gx + gw < tx or
        gy + gh < ty or
        gx > tx + tw or
        gy > ty + th
    )

for sample in dataset.iter_samples(autosave=True, progress=True):
    if sample.roi_grid is None:
        continue

    # For each tile, find overlapping dugongs
    for tile in sample.roi_grid.detections:
            tile_bbox = tile.bounding_box
            tile["dugongs"] = []
            for det in sample.ground_truth.detections:
                gt_bbox = det.bounding_box
                if bbox_overlaps(gt_bbox, tile_bbox):
                    clipped = clip_bbox(gt_bbox, tile_bbox)
                    if clipped:
                        normalized = normalize_bbox(clipped, tile_bbox)
                        tile["dugongs"].append({
                            "bbox_translated": normalized,
                            "dugong_id": det.id
                        })

    sample.save()

   0% |\--------------|   12/2755 [437.5ms elapsed, 1.7m remaining, 27.4 samples/s] 

 100% |███████████████| 2755/2755 [1.5m elapsed, 0s remaining, 19.9 samples/s]      


## convert to patches

In [ ]:
## find the ids to be converted
negative_list_id = []
positive_list_id = []

random.seed(42)
for parent_id in dataset.values("id"):

    parent_sample_view = dataset.select(parent_id)

    # 2. Filter the detections within the "roi_grid" field
    # This keeps only the detection objects where 'contains_dugong' is False
    negative = parent_sample_view.filter_labels(
        "roi_grid", 
        F("contains_dugong") == False
    )

    positive = parent_sample_view.filter_labels(
        "roi_grid", 
        F("contains_dugong") == True
    )
    grid_roi_ids_neg = negative.values("roi_grid.detections.id", unwind=True)
    grid_roi_ids_pos = positive.values("roi_grid.detections.id", unwind=True)

    ## randomly select 2 images of the given
    random.choices(grid_roi_ids_neg, k=2)

    negative_list_id.extend(random.choices(grid_roi_ids_neg, k=2))
    positive_list_id.extend(grid_roi_ids_pos)

len(negative_list_id), len(positive_list_id)

<Sample: {
    'id': '69add898cfd942e1c6b5d3ad',
    'media_type': 'image',
    'filepath': '/share/home/e2406743/dataset/dataset/NC/Flight_226/images/GH034226-619fa2d56d4d3_92.jpeg',
    'tags': ['Flight_226', 'NC', 'NC', 'train'],
    'metadata': <ImageMetadata: {
        'size_bytes': 712381,
        'mime_type': 'image/jpeg',
        'width': 2704,
        'height': 1520,
        'num_channels': 3,
    }>,
    'created_at': datetime.datetime(2026, 3, 8, 20, 14, 16, 713000),
    'last_modified_at': datetime.datetime(2026, 4, 7, 15, 41, 1, 463000),
    'region': 'NC',
    'subregion': 'NC',
    'mission_name': 'Flight_226',
    'sea_state': 0,
    'turbidity_global': 1,
    'turbidity_local': 'no',
    'sun_glitter': '0-0',
    'cloud_reflection': '0-0',
    'habitat_type': 'coral',
    'background_complexity': 'high',
    'coral': 'P',
    'sand': 'A',
    'dense_seagrass': 'A',
    'open_sea': 'P',
    'sparse_seagrass': 'A',
    'ground_truth': <Detections: {
        'detections':

### continuar aqui

In [ ]:
##
# Combine all selected IDs
all_selected_ids = negative_list_id + positive_list_id

# Filter the patches view to include ONLY these tiles
export_view = tiles_view.select(all_selected_ids)

print(f"Total tiles to export: {len(export_view)}")
# Verify the balance
print(export_view.count_values('roi_grid.detections.contains_dugong'))



# create a Patches View
# a 'virtual' dataset where each sample is one 640x640 tile
tiles_view = dataset.to_patches("roi_grid",
                                 other_fields =['id','filepath',
                                                "ground_truth",
                                                'region','subregion','mission_name','stratify_key'],
                                 keep_label_lists=True
                                  )


print(tiles_view.count_values('roi_grid.detections.contains_dugong'))


{False: 71785, True: 7253}


In [ ]:
negative_list_id = []
positive_list_id = []

random.seed(42)
for parent_id in dataset.values("id"):

    parent_sample_view = dataset.select(parent_id)

    # 2. Filter the detections within the "roi_grid" field
    # This keeps only the detection objects where 'contains_dugong' is False
    negative = parent_sample_view.filter_labels(
        "roi_grid", 
        F("contains_dugong") == False
    )

    positive = parent_sample_view.filter_labels(
        "roi_grid", 
        F("contains_dugong") == True
    )
    grid_roi_ids_neg = negative.values("roi_grid.detections.id", unwind=True)
    grid_roi_ids_pos = positive.values("roi_grid.detections.id", unwind=True)

    ## randomly select 2 images of the given
    random.choices(grid_roi_ids_neg, k=2)

    negative_list_id.extend(random.choices(grid_roi_ids_neg, k=2))
    positive_list_id.extend(grid_roi_ids_pos)

len(negative_list_id), len(positive_list_id)

(5510, 7253)

## export 

In [ ]:
import os
import json
from PIL import Image

def bbox_to_yolo(bbox):
    x, y, w, h = bbox
    return [x + w / 2, y + h / 2, w, h]


def export_patch(sample, output_dir):
    stem = f"{sample.label}_{sample.id}"
    img_path  = os.path.join(output_dir, "images",   f"{stem}.jpg")
    txt_path  = os.path.join(output_dir, "labels",   f"{stem}.txt")
    json_path = os.path.join(output_dir, "metadata", f"{stem}.json")

    for path in [img_path, txt_path, json_path]:
        os.makedirs(os.path.dirname(path), exist_ok=True)

    # --- crop & save tile image ---
    with Image.open(sample["filepath"]) as img:  # filepath = source full image
        W, H = img.size
        x, y, w, h = sample.bounding_box
        crop = img.crop((int(x*W), int(y*H), int((x+w)*W), int((y+h)*H)))
        crop = crop.resize((640, 640), Image.BILINEAR)
        crop.save(img_path, quality=95)

    # --- YOLO label ---
    lines = [
        "0 {:.6f} {:.6f} {:.6f} {:.6f}".format(*bbox_to_yolo(d["bbox_translated"]))
        for d in (sample.dugongs or [])
    ]
    with open(txt_path, "w") as f:
        f.write("\n".join(lines))

    # --- metadata ---
    meta = {
        "tile_id":        sample.id,
        "tile_label":     sample.label,
        "source_image":   sample.filepath,
        "tile_bbox":      sample.bounding_box,
        "region":         sample.region,
        "subregion":      sample.subregion,
        "mission_name":   sample.mission_name,
        "stratify_key":   sample.stratify_key,
        "contains_dugong": sample.contains_dugong,
        "dugongs":        sample.dugongs or [],
    }
    with open(json_path, "w") as f:
        json.dump(meta, f, indent=2)

In [ ]:
import os
import json
import random
from PIL import Image
from collections import defaultdict
import fiftyone as fo

# --- HELPER FUNCTIONS ---
def bbox_to_yolo(bbox):
    x, y, w, h = bbox
    # Convert [top_left_x, top_left_y, width, height] to [center_x, center_y, width, height]
    return [x + w / 2, y + h / 2, w, h]

# --- THE SAMPLING LOGIC ---
def get_balanced_patch_ids(tiles_view, negs_per_parent=2):
    random.seed(42)
    
    # 1. Map all patches to their parent images
    # sample_id is the hidden link to the 4000px source image
    all_patch_data = tiles_view.values(["id", "sample_id", "roi_grid.contains_dugong"])
    
    pos_ids = []
    parent_to_negs = defaultdict(list)
    
    for p_id, parent_id, is_pos in all_patch_data:
        if is_pos:
            pos_ids.append(p_id)
        else:
            parent_to_negs[parent_id].append(p_id)
            
    # 2. Sample 2 negatives from EACH parent image that exists in the dataset
    sampled_neg_ids = []
    for parent_id, neg_list in parent_to_negs.items():
        num_to_take = min(len(neg_list), negs_per_parent)
        sampled_neg_ids.extend(random.sample(neg_list, num_to_take))
        
    return pos_ids + sampled_neg_ids

# --- THE MODIFIED EXPORT LOOP ---
def run_thesis_export(dataset, output_dir):
    # Refresh view to ensure custom fields are visible
    tiles_view = dataset.to_patches("roi_grid")
    
    # Get our balanced list of IDs
    keep_ids = get_balanced_patch_ids(tiles_view)
    export_view = tiles_view.select(keep_ids)
    
    print(f"Starting export of {len(export_view)} tiles...")

    for sample in export_view.iter_samples(progress=True):
        # --- Paths ---
        # Using roi_grid.label (e.g., tile_0_540) ensures unique filenames
        stem = f"{sample.roi_grid.label}_{sample.id}"
        img_path  = os.path.join(output_dir, "images",   f"{stem}.jpg")
        txt_path  = os.path.join(output_dir, "labels",   f"{stem}.txt")
        json_path = os.path.join(output_dir, "metadata", f"{stem}.json")

        for path in [img_path, txt_path, json_path]:
            os.makedirs(os.path.dirname(path), exist_ok=True)

        # --- Crop & Save Tile Image ---
        # filepath is the path to the 4000px source image
        with Image.open(sample.filepath) as img:
            W, H = img.size
            # Get the tile location from the roi_grid field
            x, y, w, h = sample.roi_grid.bounding_box
            
            left, top = int(x * W), int(y * H)
            right, bottom = int((x + w) * W), int((y + h) * H)
            
            crop = img.crop((left, top, right, bottom))
            crop = crop.resize((640, 640), Image.BILINEAR)
            crop.save(img_path, quality=95)

        # --- YOLO Label (RT-DETR compatible) ---
        # Note: Accessing your 'dugongs' list from the roi_grid attribute
        dugong_list = sample.roi_grid.get("dugongs", [])
        lines = [
            "0 {:.6f} {:.6f} {:.6f} {:.6f}".format(*bbox_to_yolo(d["bbox_translated"]))
            for d in dugong_list
        ]
        with open(txt_path, "w") as f:
            f.write("\n".join(lines))

        # --- Metadata (For your Thesis Analysis) ---
        meta = {
            "tile_id":         str(sample.id),
            "parent_id":       str(sample.sample_id),
            "tile_label":      sample.roi_grid.label,
            "source_image":    sample.filepath,
            "region":          sample.region,
            "mission_name":    sample.mission_name,
            "background":      sample.background_complexity,
            "contains_dugong": sample.roi_grid.contains_dugong,
            "dugongs":         dugong_list,
        }
        with open(json_path, "w") as f:
            json.dump(meta, f, indent=2)

# --- EXECUTE ---
run_thesis_export(dataset, "./Dugong_RTDETR_Export_Run1")

In [ ]:
import random
from collections import defaultdict
from fiftyone import ViewField as F

# 1. Isolate Positives and Negatives using the internal list path
positives = tiles_view.match(F("roi_grid.detections.contains_dugong").contains(True))
negatives = tiles_view.match(
    F("roi_grid.detections").filter(F("contains_dugong") == True).length() == 0
)
# 2. Extract IDs and the Parent Image IDs
pos_patch_ids = positives.values("id")
pos_parent_ids = positives.values("sample_id")

neg_patch_ids = negatives.values("id")
neg_parent_ids = negatives.values("sample_id")

# 3. Create the Map: Parent Image -> List of Negative Tiles
parent_to_negs = defaultdict(list)
for p_id, patch_id in zip(neg_parent_ids, neg_patch_ids):
    parent_to_negs[p_id].append(patch_id)

# 4. Contextual Sampling (3 Negatives per Parent with a Positive)
unique_parents = list(set(pos_parent_ids))
sampled_neg_ids = []
random.seed(42)

for p_id in unique_parents:
    potential = parent_to_negs.get(p_id, [])
    if potential:
        num_to_take = min(len(potential), 3)
        sampled_neg_ids.extend(random.sample(potential, num_to_take))

# 5. Combine and Create Export View
export_view = tiles_view.select(list(pos_patch_ids) + sampled_neg_ids)

print("-" * 30)
print(f"Final Selection for Thesis:")
print(f"Positive Patches: {len(pos_patch_ids)}")
print(f"Contextual Negative Patches: {len(sampled_neg_ids)}")
print(f"Total Export Count: {len(export_view)}")

AttributeError: 'ViewExpression' object has no attribute 'negate'

In [ ]:
random.seed(42)

# 1. Separate your patches
positives = tiles_view.match({"roi_grid.detections.contains_dugong": True})
negatives = tiles_view.match({"roi_grid.detections.contains_dugong": False})

# 2. Define your "Background Budget" per image
# For every image that has a dugong, how many empty patches do we want?
neg_per_parent = 2

# 3. Get the list of parent images that actually contain dugongs
# This ensures we are picking background from 'relevant' environments
parent_ids_with_positives = positives.distinct("id")

neg_sample_ids = []

print(f"Sampling {neg_per_parent} negatives from {len(parent_ids_with_positives)} parent images...")

for parent_id in parent_ids_with_positives:
    # Find all negative patches belonging to THIS parent image
    parent_neg_view = negatives.match(F("id") == parent_id)
    
    # Get their IDs
    all_neg_ids = parent_neg_view.values("ground_truth.detections.sample_id")
    
    if all_neg_ids:
        # Take up to neg_per_parent patches
        num_to_sample = min(len(all_neg_ids), neg_per_parent)
        sampled_ids = random.sample(all_neg_ids, num_to_sample)
        neg_sample_ids.extend(sampled_ids)

# 4. Create the final view for export
export_view = positives + tiles_view.select(neg_sample_ids)

print("-" * 30)
print(f"Positive patches: {len(positives)}")
print(f"Negative patches (Contextual): {len(neg_sample_ids)}")
print(f"Total patches for Run: {len(export_view)}")

ValueError: PatchesView has no field 'ground_truth'

In [61]:
parent_ids_with_positives

[]

In [56]:
tiles_view.match(F("roi_grid.detections.contains_dugong") == True)

Dataset:     dugong
Media type:  image
Num patches: 0
Patch fields:
    id:               fiftyone.core.fields.ObjectIdField
    sample_id:        fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    roi_grid:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    region:           fiftyone.core.fields.StringField
    subregion:        fiftyone.core.fields.StringField
    mission_name:     fiftyone.core.fields.StringField
    stratify_key:     fiftyone.core.fields.StringField
View stages:
    1. ToPatches(field='roi_grid', config={'keep_label_lists': True, 'other_fields': ['id', 'filepath', 'region', ...]})
    2. 

In [ ]:
## export all true images and 
import os
import json
from PIL import Image
import fiftyone as fo
from tqdm import tqdm

def export_metadata_manifest(dataset, run_number, output_base_dir, bg_ratio=2):
    """
    Physically crops tiles and creates a JSON manifest with full metadata.
    """
    run_dir = os.path.join(output_base_dir, f"run_{run_number}")
    images_dir = os.path.join(run_dir, "patches")
    os.makedirs(images_dir, exist_ok=True)

    manifest = []
    
    # We process Train, Val, and Test separately
    for split in ["train", "val", "test"]:
        split_tag = f"{split}_run{run_number}"
        view = dataset.match_tags(split_tag)
        
        print(f"Processing {split} for Run {run_number}...")

        for sample in view.progress_view():
            # Load the original high-res image once per sample
            with Image.open(sample.filepath) as img:
                w, h = img.size
                
                for i, tile in enumerate(sample.roi_grid.detections):
                    # 1. Determine if we should export this tile
                    is_positive = tile.get("contains_dugong", False)
                    
                    # Logic: Always keep positives. For negatives in training, 
                    # we can sub-sample later in the dataloader using the manifest.
                    # For now, let's export all positives and a balanced set of negatives.
                    if not is_positive and split == "train":
                        # Simple skip logic to keep the disk space manageable
                        if i % 10 != 0: continue 

                    # 2. Calculate Pixel Coordinates for the Crop
                    tx, ty, tw, th = tile.bounding_box
                    left = tx * w
                    top = ty * h
                    right = (tx + tw) * w
                    bottom = (ty + th) * h
                    
                    # 3. Perform the physical crop
                    patch_filename = f"{sample.id}_tile_{i}.jpg"
                    patch_path = os.path.join(images_dir, patch_filename)
                    
                    if not os.path.exists(patch_path):
                        tile_img = img.crop((left, top, right, bottom))
                        tile_img.convert("RGB").save(patch_path, quality=95)

                    # 4. Build the Metadata Entry
                    entry = {
                        "patch_path": os.path.abspath(patch_path),
                        "split": split,
                        "parent_id": str(sample.id),
                        "region": sample.region,
                        "mission": sample.mission_name,
                        "background_complexity": sample.background_complexity,
                        "contains_dugong": is_positive,
                        "labels": tile.get("dugongs", []) # This contains your bbox_translated!
                    }
                    manifest.append(entry)

    # Save the manifest as the 'Source of Truth' for your Dataloader
    manifest_path = os.path.join(run_dir, f"manifest_run_{run_number}.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=4)
        
    print(f"Export complete. Manifest saved at: {manifest_path}")

# Execute
export_metadata_manifest(dataset, run_number=1, output_base_dir="./Dugong_Thesis_Exports")

In [40]:
tiles_view.count_values('roi_grid.detections.contains_dugong')

{True: 7253, False: 71785}

In [ ]:
tiles_view.match(
    F("roi_grid.detections.contains_dugong").)


SyntaxError: expression cannot contain assignment, perhaps you meant "=="? (3892188808.py, line 2)

In [9]:
len(tiles_with_dugongs)

79038

In [10]:
len(background_tiles)

0

In [11]:
tiles_view

Dataset:     dugong
Media type:  image
Num patches: 79038
Patch fields:
    id:               fiftyone.core.fields.ObjectIdField
    sample_id:        fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    roi_grid:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detection)
    region:           fiftyone.core.fields.StringField
    subregion:        fiftyone.core.fields.StringField
    mission_name:     fiftyone.core.fields.StringField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    stratify_key:     fiftyone.core.fields.StringField
View stages:
    1. ToPatches(field='roi